# Voting Ballots + EMNIST dataset

In [1]:
import numpy as np
import math
import random
import matplotlib.pyplot as plt

from PIL import Image
from skimage.io import imread
#from rembg import remove
import albumentations as A
import cv2

import os
import os.path as osp

from tqdm import tqdm

from pathlib import Path

/home/ahabanen/.local/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [4]:
def create_dataset_folders(home_path, dataset_name, dataset_split):

    dataset_folder = osp.join(home_path, dataset_name) 
    if not os.path.exists(dataset_folder):
        os.makedirs(dataset_folder)

    images_folder = osp.join(dataset_folder, "images") 
    if not os.path.exists(images_folder):
        os.makedirs(images_folder)

    labels_folder = osp.join(dataset_folder, "labels") 
    if not os.path.exists(labels_folder):
        os.makedirs(labels_folder)

    images_split_folder = osp.join(images_folder, dataset_split) 
    if not os.path.exists(images_split_folder):
        os.makedirs(images_split_folder)

    labels_split_folder = osp.join(labels_folder, dataset_split) 
    if not os.path.exists(labels_split_folder):
        os.makedirs(labels_split_folder)

    return images_split_folder, labels_split_folder
    

In [5]:
def reading_in_bbox_txt_file(digit_bbox_path, isBallotBox) :
    bboxes = []

    with open(digit_bbox_path, "r") as file:
        while True:
            content=file.readline()
            if not content:
                break
            elements = [ float(x) for x in content.strip().split(" ") ]
            if isBallotBox:
                elements.append('ballot_box')
            bboxes.append(elements)

    return bboxes


In [6]:
def reading_in_random_digit(image_path, bbox_path, digits_filenames) :

    random_image = random.randint(0, len(digits_filenames)-1)
    file_name = str(digits_filenames[random_image].split(".")[0])
    digit_image_path = osp.join(image_path, file_name + '.png')
    digit_bbox_path = osp.join(bbox_path, file_name + '.txt')
    
    digit = cv2.cvtColor(cv2.imread(digit_image_path), cv2.COLOR_BGR2RGBA)
    
    digit_bboxes = reading_in_bbox_txt_file(digit_bbox_path, False)

    return digit, digit_bboxes

In [7]:
def find_area_for_digit_in_voting_ballot_box(ballot, ballot_bbox):

    # Voting ballot image size
    image_width = ballot.size[0]
    image_height = ballot.size[1]
    
    # Voting ballot box info
    box_x = ballot_bbox[0][0] * image_width
    box_y = ballot_bbox[0][1] * image_height
    box_width = ballot_bbox[0][2] * image_width
    box_height = ballot_bbox[0][3] * image_height
    
    # Based on digit and voting ballot box ratios we choose if we resize the digit image based on the box width or height
    if 3 > box_width/box_height:
        # Based on box width
        small_width = int(box_width*0.9)
        small_height = int(small_width / 3)
    else:
        # Based on box height
        small_height = int(box_height*0.85)
        small_width = small_height*3

    return box_x, box_y, small_width, small_height

In [8]:
# https://stackoverflow.com/questions/48395434/how-to-crop-or-remove-white-background-from-an-image
  
def remove_background(image):
    pixels = image.getdata()
    newData = []
  
    for item in pixels:
        if item[0] <= 255 and item[0] >= 200 and item[1] <= 255 and item[1] >= 200 and item[2] <= 255 and item[2] >= 200:
            newData.append((255, 255, 255, 0))
        else:
            newData.append(item)

    image.putdata(newData)
    return image

In [9]:
def get_resized_digit_transform(width, height):
    return A.Compose([
        A.Resize(width = width, height = height)
    ])

In [10]:

def get_transformed_image(crop_width, crop_height):
    return A.Compose([
        # Pixel-level transforms
        A.RandomBrightnessContrast(p=0.8),
        A.HueSaturationValue(p=0.8, hue_shift_limit=(-10, 10), sat_shift_limit=(-10,10), val_shift_limit=(-10,10)),
        A.GaussianBlur(p=0.1, blur_limit=(3, 5)),
        # Spatial-level transforms
        A.Perspective(p=1, ),
        A.RandomSizedBBoxSafeCrop(p=1, height=360, width=640),
        A.Resize(height=1080, width=1920, p=1),
    ], bbox_params=A.BboxParams(format='yolo', min_visibility=0.8, label_fields=['class_labels']))


In [11]:
def find_new_digit_bboxes_on_ballot(digit_bboxes, xstart, ystart, ballot, digit):
    new_bboxes = []
    labels = []

    for i in range(len(digit_bboxes)):
        bbox = [(digit_bboxes[i][1] * digit.size[0] + xstart) / ballot.size[0],
                (digit_bboxes[i][2] * digit.size[1] + ystart) / ballot.size[1],
                (digit_bboxes[i][3] * digit.size[0]) / ballot.size[0],
                (digit_bboxes[i][4] * digit.size[1]) / ballot.size[1]
               ]
        labels.append(digit_bboxes[i][0])
        new_bboxes.append(bbox)

    return new_bboxes, labels
    

In [12]:
def save_bboxes_into_txt(path, bboxes, labels):
    with open(path, "w") as file:
        for i in range(len(bboxes)):
            file.write(str(labels[i]) + " " + str(bboxes[i][0]) + " " + str(bboxes[i][1]) + " " + str(bboxes[i][2]) + " " + str(bboxes[i][3]) + "\n")
        file.close()       

In [ ]:

id_length = 5
parent_path = str(Path.cwd().parent)
home_path = parent_path + "/datasets"
dataset_splits = ["train", "test", "val"]
dataset_ballots = [34, 5, 5]


for s in range(2, len(dataset_splits)):

    split = dataset_splits[s]
    nr_of_ballots = dataset_ballots[s]

    ballots_images_path = home_path + "/ballot_images/images"
    ballots_bboxes_path = home_path + "/ballot_images/labels"
    
    digits_images_path = home_path + "/EMNIST_3digits/images/" + split
    digits_bboxes_path = home_path + "/EMNIST_3digits/labels/" + split
    
    augmented_ballots_images_path, augmented_ballots_bboxes_path = create_dataset_folders(home_path, "voting_ballots_+_digits", split)
    
    digits_file_names = os.listdir(digits_images_path)

    print("Starting creating for new split: " + split)
    
    for i in tqdm(range(40, 45)): # Going through all ballots

        print("New ballot")
        
        ballot_image_path = osp.join(ballots_images_path, 'ballot_{}_a.png'.format(i))
        ballot_bbox_path = osp.join(ballots_bboxes_path, 'ballot_{}_a.txt'.format(i))
        ballot = Image.open(ballot_image_path)
        ballot_bbox = reading_in_bbox_txt_file(ballot_bbox_path, True)
    
        ballot_img_width, ballot_img_height = ballot.size
        image_transform = get_transformed_image(ballot_img_width*0.7, ballot_img_height*0.7)
        
        box_x, box_y, box_width, box_height = find_area_for_digit_in_voting_ballot_box(ballot, ballot_bbox)
        xstart = int(box_x - box_width/2)
        ystart = int(box_y - box_height/2)
        resizing = get_resized_digit_transform(box_width, box_height)
    
        nr_of_aug_per_ballot = 1
    
        for j in tqdm(range(1, 61)): # Nr of different digits per ballot
    
            ballot = Image.open(ballot_image_path)
            # Getting random digit
            digit, digit_bboxes = reading_in_random_digit(digits_images_path, digits_bboxes_path, digits_file_names)
            
            # Resizing the digit based on ballot box size
            digit_resized = resizing(image=digit)
            digit = digit_resized['image']
            digit = Image.fromarray(digit, 'RGBA')
            
            # Removing background
            digit_bg_removed = remove_background(digit)
            
            # Placing the digit onto the ballot box area.
            ballot.paste(digit_bg_removed, (xstart, ystart), digit_bg_removed)
    
            # Some transforms need 'RGB' image
            ballot = ballot.convert('RGB')
    
            # Recalculating digit bbox info
            digit_bboxes_on_ballot, digit_labels = find_new_digit_bboxes_on_ballot(digit_bboxes, xstart, ystart, ballot, digit)
        
    
            for k in range(1, 6): # Nr of augmentations per digit on one ballot
        
                # Transform the image
                transformed = image_transform(image=np.array(ballot), bboxes=digit_bboxes_on_ballot, class_labels=digit_labels)
                transformed_image = transformed['image']
                transformed_bboxes = transformed['bboxes']
                transformed_labels = transformed['class_labels']
                
                image_id = '0' * (id_length - len(str(nr_of_aug_per_ballot))) + str(nr_of_aug_per_ballot)
                
                # Save image and digit bounding boxes
                image_path = osp.join(augmented_ballots_images_path, 'ballots_merge_{}_{}_{}.png'.format(i, j, k))
                tranformed_ballot = Image.fromarray(np.uint8(transformed_image)).convert('RGBA')
                tranformed_ballot.save(image_path, format='png')
        
                bbox_path = osp.join(augmented_ballots_bboxes_path, 'ballots_merge_{}_{}_{}.txt'.format(i, j, k))
                save_bboxes_into_txt(bbox_path, transformed_bboxes, transformed_labels)
    
                nr_of_aug_per_ballot += 1


        

## Visulizing

In [ ]:
ballot_id = 1
image_id = '0001'

parent_path = str(Path.cwd().parent)
image_path = parent_path + "/datasets/voting_ballots_+_digits/images/train/ballots_merge_"+str(ballot_id)+"_"+str(image_id)+".png" # replace with existing file name
label_path = parent_path + "/datasets/voting_ballots_+_digits/labels/train/ballots_merge_"+str(ballot_id)+"_"+str(image_id)+".txt" # replace with existing file name


img = imread(image_path)

In [15]:
def show_image_with_bounding_box (xcentre, ycentre, width, heigth, image_width, image_height):

    xmin = (xcentre - width/2) * image_width
    ymin = (ycentre - heigth/2) * image_height
    xmax = (xcentre + width/2) * image_width
    ymax = (ycentre + heigth/2) * image_height

    plt.plot([xmin, xmin], [ymin, ymax], '-', color = 'red' ) # Left edge
    plt.plot([xmax, xmax], [ymin, ymax], '-', color = 'red') # Right edge
    plt.plot([xmin, xmax], [ymin, ymin], '-', color = 'red') # Top edge
    plt.plot([xmin, xmax], [ymax, ymax], '-', color = 'red') # Bottom edge
    

In [ ]:
label_bboxes = reading_in_bbox_txt_file(label_path, False)

plt.imshow(img)

for i in range(len(label_bboxes)):
    show_image_with_bounding_box(label_bboxes[i][1], label_bboxes[i][2], label_bboxes[i][3], label_bboxes[i][4], img.shape[1], img.shape[0])

plt.axis('on')
plt.show()